# Elliptic Dataset: Static Graph Neural Network

Can learned message passing outperform engineered graph features?

Do GCN and GraphSAGE still benefit from hand-engineered graph features, or can message passing recover enough graph structure by itself?

Goal: Train a static Graph Convolutional Network on the full Elliptic graph, while evaluating with temporal train/validation/test masks.

In [70]:
!pip install torch_geometric

In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import networkx as nx

import torch
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv

from sklearn.metrics import classification_report, confusion_matrix

import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import os

In [72]:
# Keep consistency
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [73]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [74]:
df_classes = pd.read_csv('/content/drive/MyDrive/elliptic_txs_classes.csv')
df_classes['class'] = pd.to_numeric(df_classes['class'], errors = 'coerce')
df_classes['class'] = df_classes['class'].map({1: 'illicit', 2: 'licit'}).fillna('unknown')

df_edgelist = pd.read_csv('/content/drive/MyDrive/elliptic_txs_edgelist.csv')
df_edgelist = df_edgelist.rename(columns={'txId1': 'source', 'txId2': 'target'})


df_features = pd.read_parquet("/content/drive/MyDrive/Elliptic/processed/nodes/clean.parquet")

comparison_03 = pd.read_csv(
    "/content/drive/MyDrive/Elliptic/results/03_graph_feature_engineering_comparison.csv"
)

In [75]:
comparison_03.head(6)

,Feature Set,Model,Num Features,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Difference vs. Original,Runtime Minutes
0,Original Features,Random Forest,169,0.990339,0.994975,0.885906,0.937278,0.000000,0.694
1,Node2Vec Embeddings,Random Forest,201,0.989245,0.989899,0.876957,0.930012,-0.007266,0.933
2,DeepWalk Embeddings,Random Forest,201,0.989245,0.992386,0.874720,0.929845,-0.007433,0.927
3,Node2Vec Embeddings,Logistic Regression,201,0.784360,0.269424,0.961969,0.420950,0.001706,0.094
4,Original Features,Logistic Regression,169,0.784360,0.268553,0.955257,0.419244,0.000000,0.176
5,DeepWalk Embeddings,Logistic Regression,201,0.770142,0.255995,0.955257,0.403783,-0.015461,0.102


In [76]:
print(df_features.shape)
print(df_features.columns[:20])
print(df_features[['txId', 'time_step', 'label']].head())

(203769, 172)
Index(['txId', 'time_step', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11',
       '12', '13', '14', '15', '16', '17', '18', '19'],
      dtype='object')
        txId  time_step    label
0  230425980          1  unknown
1    5530458          1  unknown
2  232022460          1  unknown
3  232438397          1    licit
4  230460314          1  unknown


### Temporal Mask and Temporal Data Split

Although this notebook investigates static Graph Neural Networks (GCNs and GraphSAGE), the models are not evaluated using a random train/test split. Instead, the evaluation follows the same temporal validation protocol used throughout this project to better reflect a real-world fraud detection scenario.

The Elliptic Bitcoin dataset spans 49 temporal snapshots. To avoid training on future transactions, the data is divided chronologically:

Training: Time steps 1–34 (29,894 labeled transactions)
Validation: Time steps 35–39 (5,486 labeled transactions)
Testing: Time steps 40–49 (11,184 labeled transactions)
The graph itself is treated as a single static network, meaning the graph structure does not change during training. However, the training, validation, and testing masks are based on transaction time, ensuring that model performance is evaluated on future observations rather than on randomly selected nodes.

The training set is used to optimize the model parameters (weights). The validation set is used during model development to monitor performance, compare architectures (such as GCN versus GraphSAGE), and guide choices such as the number of training epochs or hyperparameters without using the final test data. Finally, the test set is evaluated only after model development is complete to provide an unbiased estimate of how well the model generalizes to unseen future transactions.

This temporal evaluation protocol helps reduce overly optimistic performance estimates that can arise when future information is inadvertently incorporated into the training process.

In [77]:
def build_pyg_data(df_features, df_edgelist, feature_columns):
    # Map transaction IDs to sequential indices for edge list
    txid_to_idx = {
        txid: idx for idx, txid in enumerate(df_features['txId'])
    }

    # Build edge index
    edge_index = torch.tensor([
        df_edgelist['source'].map(txid_to_idx).values,
        df_edgelist['target'].map(txid_to_idx).values
    ], dtype=torch.long)

    # Make undirected for basic GCN/GraphSAGE
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

    # Build features (x)
    x = torch.tensor(df_features[feature_columns].values, dtype=torch.float)

    # Build labels (y)
    label_map = {
        'licit': 0,
        'illicit': 1,
        'unknown': -1
    }
    y = torch.tensor(
        df_features['label'].map(label_map).values,
        dtype=torch.long
    )

    # Create temporal masks
    time_step = torch.tensor(df_features['time_step'].values)
    known_mask = y != -1
    train_mask = known_mask & (time_step <= 34)
    val_mask = known_mask & ((time_step >= 35) & (time_step <= 39))
    test_mask = known_mask & (time_step >= 40)

    # Create PyG Data object
    data = Data(
        x=x,
        edge_index=edge_index,
        y=y
    )

    return data, train_mask, val_mask, test_mask, time_step

### Features Decision

This notebook uses the original Elliptic transaction features together with the engineered graph features generated in previous notebooks: degree, clustering coefficient, PageRank, and eigenvector centrality. These features are included in df_features and are used as the primary input feature set for the initial GCN and GraphSAGE experiments.

This provides a strong static GNN baseline because the models receive both node-level transaction features and hand-engineered graph structure. A later ablation removes the engineered graph features to test whether message passing alone can recover similar structural information from the graph.

In [78]:
graph_cols = ['degree',
              'clustering_coefficient',
              'pagerank',
              'eigenvector_centrality']

features_cols = [col for col in df_features.columns
                 if col not in ['txId', 'time_step', 'label']]

print("Number of features:", len(features_cols))

Number of features: 169


In [79]:
# Define feature columns without hand-crafted graph features
features_cols_no_graph = [col for col in df_features.columns
                          if col not in ['txId', 'time_step', 'label'] and col not in graph_cols]

print("Number of features (original only):", len(features_cols_no_graph))

Number of features (original only): 165


Now, we will create two PyTorch Geometric `Data` objects:
1. `data_full`: Using all original features plus the hand-crafted graph features.
2. `data_no_graph`: Using only the original features, without the hand-crafted graph features.

This will allow us to directly compare the impact of these engineered graph features on GNN performance.

In [80]:
# Create data object with all features (original + graph features)
data_full, train_mask_full, val_mask_full, test_mask_full, time_step_full = build_pyg_data(
    df_features, df_edgelist, features_cols
)

# Create data object with only original features
data_no_graph, train_mask_no_graph, val_mask_no_graph, test_mask_no_graph, time_step_no_graph = build_pyg_data(
    df_features, df_edgelist, features_cols_no_graph
)

print("\nData object (full features):")
print(data_full)
print(f"Train mask (full features): {train_mask_full.sum().item()} nodes")
print(f"Validation mask (full features): {val_mask_full.sum().item()} nodes")
print(f"Test mask (full features): {test_mask_full.sum().item()} nodes")

print("\nData object (no graph features):")
print(data_no_graph)
print(f"Train mask (no graph features): {train_mask_no_graph.sum().item()} nodes")
print(f"Validation mask (no graph features): {val_mask_no_graph.sum().item()} nodes")
print(f"Test mask (no graph features): {test_mask_no_graph.sum().item()} nodes")


Data object (full features):
Data(x=[203769, 169], edge_index=[2, 468710], y=[203769])
Train mask (full features): 29894 nodes
Validation mask (full features): 5486 nodes
Test mask (full features): 11184 nodes

Data object (no graph features):
Data(x=[203769, 165], edge_index=[2, 468710], y=[203769])
Train mask (no graph features): 29894 nodes
Validation mask (no graph features): 5486 nodes
Test mask (no graph features): 11184 nodes


## GCN - Graph Convolutional Network

A Graph Convolutional Network is a neural network designed for graph-structured data. Unlike a standard neural network, which treats each row independently, a GCN updates each node representation by combining its own features with information from neighboring nodes.

In this notebook, the GCN uses transaction-level features together with the Elliptic graph structure. Each layer performs message passing: neighboring node information is aggregated, transformed by learnable weights, and passed through a nonlinear activation function. This allows the model to learn node representations that depend on both transaction features and graph connectivity.

This static GCN is different from Node2Vec and DeepWalk. Node2Vec and DeepWalk first create embeddings using random walks, then pass those embeddings into a separate machine learning model. The GCN learns node representations and predictions jointly during training.

Because the Elliptic graph is treated as one static network, temporal risk is controlled through chronological train, validation, and test masks rather than through dynamic graph construction.

### GCN Model

In [81]:
class GCN(torch.nn.Module):
  def __init__(self, in_channels, hidden_channels=64, out_channels=2, dropout=0.5):
    super().__init__()
    self.conv1 = GCNConv(in_channels, hidden_channels)
    self.conv2 = GCNConv(hidden_channels, out_channels)
    self.dropout = dropout

  def forward(self, data):
    x, edge_index = data.x, data.edge_index
    x = self.conv1(x, edge_index)
    x = F.relu(x)
    x = F.dropout(x, p=self.dropout, training=self.training)
    x = self.conv2(x, edge_index)
    return x

## GraphSAGE

GraphSAGE (which stands for Graph SAmple and aggreGatE) is a machine learning framework used to generate vector representations (embeddings) for nodes in large-scale graphs.

GraphSAGE is a framework that generates embeddings by learning differentiable aggregator functions instead of optimizing distinct node-specific vectors. The process begins by sampling a fixed-size, localized neighborhood around each target node to maintain computational efficiency across large networks. Next, the algorithm aggregates feature information from these sampled neighbors using mathematical functions like Mean, LSTM, or Pooling operators. Finally, this aggregated neighborhood representation is concatenated with the target node's current features and passed through a fully connected layer to compute the updated node embedding.

GCN because it is still a static GNN, but it uses neighborhood aggregation instead of the GCN normalization-style convolution. PyTorch Geometric’s `SAGEConv` uses neighborhood aggregation such as `"mean"` by default.

In [82]:
class GraphSAGE(torch.nn.Module):
  def __init__(self, in_channels, hidden_channels=64, out_channels=2, dropout=0.5):
    super().__init__()

    self.sage1 = SAGEConv(in_channels, hidden_channels, aggr='mean')
    self.sage2 = SAGEConv(hidden_channels, out_channels, aggr='mean')

    self.dropout = dropout

  def forward(self, data):
    x, edge_index = data.x, data.edge_index

    x = self.sage1(x, edge_index)
    x = F.relu(x)
    x = F.dropout(x, p=self.dropout, training=self.training)

    x = self.sage2(x, edge_index)

    return x

In [83]:
# training
def evaluate_gnn_model(model_name, feature_set_name, model, data, train_mask, val_mask, epochs=200):
    start_time = time.perf_counter()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data)
        loss = F.cross_entropy(out[train_mask], data.y[train_mask])

        loss.backward()
        optimizer.step()

    runtime_seconds = time.perf_counter() - start_time

    model.eval()
    with torch.no_grad():
        out = model(data)
        # Evaluate on validation set for model comparison during development
        val_preds = out[val_mask].argmax(dim=1).cpu().numpy()
        y_val = data.y[val_mask].cpu().numpy()

    return {
        "Feature Set": feature_set_name,
        "Model": model_name,
        "Num Features": data.x.shape[1],
        "Accuracy": accuracy_score(y_val, val_preds),
        "Illicit Precision": precision_score(y_val, val_preds, pos_label=1),
        "Illicit Recall": recall_score(y_val, val_preds, pos_label=1),
        "Illicit F1": f1_score(y_val, val_preds, pos_label=1),
        "Runtime Seconds": runtime_seconds,
        "Runtime Minutes": runtime_seconds / 60
    }

## Comparison Table

This section extends the comparison from Notebook 03 by adding GCN and GraphSAGE evaluated under the same temporal validation protocol.


### Caching GNN Experiment Results

Training GCN and GraphSAGE models requires approximately 30 minutes for the validation experiments. To improve development efficiency, this notebook uses a simple caching mechanism for the validation-stage comparison results.

When the notebook is executed, it first checks whether 04_static_gnn_results_raw.csv already exists.

If the file exists, the previously computed GCN and GraphSAGE validation results are loaded directly.
If the file does not exist, the GCN and GraphSAGE experiments are trained from scratch, the comparison results are saved to 04_static_gnn_results_raw.csv, and those results are used throughout the remainder of the notebook.

This caching mechanism only affects the validation-stage model comparison. The Final Test Set Evaluation is intentionally not cached. The models are retrained and evaluated on the unseen test set each time the notebook is executed to provide an unbiased estimate of generalization performance.

Readers who wish to reproduce the complete validation experiments can simply delete 04_static_gnn_results_raw.csv and rerun the notebook.

In [84]:
results_path = "/content/drive/MyDrive/Elliptic/results/04_static_gnn_results_raw.csv"

if os.path.exists(results_path):
    print("Found existing GNN results. Loading saved comparison table...")
    comparison_gnn_results = pd.read_csv(results_path)
else:
    print("No saved GNN results found. Running GCN and GraphSAGE experiments...")
    gnn_results = []

    # GCN results for 'Original + Graph Features'
    gcn_model_full = GCN(in_channels=data_full.x.shape[1])
    gcn_result_full = evaluate_gnn_model(
        model_name='GCN',
        feature_set_name='Original + Graph Features',
        model=gcn_model_full,
        data=data_full,
        train_mask=train_mask_full,
        val_mask=val_mask_full,
        epochs=200
    )
    gnn_results.append(gcn_result_full)

    # GCN results for 'Original Features Only'
    gcn_model_no_graph = GCN(in_channels=data_no_graph.x.shape[1])
    gcn_result_no_graph = evaluate_gnn_model(
        model_name='GCN',
        feature_set_name='Original Features Only',
        model=gcn_model_no_graph,
        data=data_no_graph,
        train_mask=train_mask_no_graph,
        val_mask=val_mask_no_graph,
        epochs=200
    )
    gnn_results.append(gcn_result_no_graph)

    # GraphSAGE results for 'Original + Graph Features'
    sage_model_full = GraphSAGE(in_channels=data_full.x.shape[1])
    sage_result_full = evaluate_gnn_model(
        model_name='GraphSAGE',
        feature_set_name='Original + Graph Features',
        model=sage_model_full,
        data=data_full,
        train_mask=train_mask_full,
        val_mask=val_mask_full,
        epochs=200
    )
    gnn_results.append(sage_result_full)

    # GraphSAGE results for 'Original Features Only'
    sage_model_no_graph = GraphSAGE(in_channels=data_no_graph.x.shape[1])
    sage_result_no_graph = evaluate_gnn_model(
        model_name='GraphSAGE',
        feature_set_name='Original Features Only',
        model=sage_model_no_graph,
        data=data_no_graph,
        train_mask=train_mask_no_graph,
        val_mask=val_mask_no_graph,
        epochs=200
    )
    gnn_results.append(sage_result_no_graph)

    comparison_gnn_results = pd.DataFrame(gnn_results)
    comparison_gnn_results.to_csv(results_path, index=False)
    print("Saved GNN results.")

display(comparison_gnn_results)

Found existing GNN results. Loading saved comparison table...


,Feature Set,Model,Num Features,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Runtime Seconds,Runtime Minutes
0,Original + Graph Features,GCN,169,0.952971,0.853933,0.510067,0.638655,368.003917,6.133399
1,Original Features Only,GCN,165,0.948232,0.802974,0.483221,0.603352,349.176522,5.819609
2,Original + Graph Features,GraphSAGE,169,0.971382,0.909605,0.720358,0.803995,518.295740,8.638262
3,Original Features Only,GraphSAGE,165,0.968647,0.874659,0.718121,0.788698,518.108137,8.635136


In [85]:
pass

In [86]:
gnn_results

[{'Feature Set': 'Original + Graph Features',
  'Model': 'GCN',
  'Num Features': 169,
  'Accuracy': 0.958075100255195,
  'Illicit Precision': 0.8703071672354948,
  'Illicit Recall': 0.5704697986577181,
  'Illicit F1': 0.6891891891891891,
  'Runtime Seconds': 375.25686739900084,
  'Runtime Minutes': 6.254281123316681},
 {'Feature Set': 'Original Features Only',
  'Model': 'GCN',
  'Num Features': 165,
  'Accuracy': 0.9564345606999636,
  'Illicit Precision': 0.8969465648854962,
  'Illicit Recall': 0.5257270693512305,
  'Illicit F1': 0.6629055007052186,
  'Runtime Seconds': 355.60681742499946,
  'Runtime Minutes': 5.926780290416658},
 {'Feature Set': 'Original + Graph Features',
  'Model': 'GraphSAGE',
  'Num Features': 169,
  'Accuracy': 0.9702880058330295,
  'Illicit Precision': 0.8717277486910995,
  'Illicit Recall': 0.7449664429530202,
  'Illicit F1': 0.8033775633293124,
  'Runtime Seconds': 535.5016962000009,
  'Runtime Minutes': 8.925028270000015},
 {'Feature Set': 'Original Featur

In [87]:
pass

In [88]:
gnn_results

[{'Feature Set': 'Original + Graph Features',
  'Model': 'GCN',
  'Num Features': 169,
  'Accuracy': 0.958075100255195,
  'Illicit Precision': 0.8703071672354948,
  'Illicit Recall': 0.5704697986577181,
  'Illicit F1': 0.6891891891891891,
  'Runtime Seconds': 375.25686739900084,
  'Runtime Minutes': 6.254281123316681},
 {'Feature Set': 'Original Features Only',
  'Model': 'GCN',
  'Num Features': 165,
  'Accuracy': 0.9564345606999636,
  'Illicit Precision': 0.8969465648854962,
  'Illicit Recall': 0.5257270693512305,
  'Illicit F1': 0.6629055007052186,
  'Runtime Seconds': 355.60681742499946,
  'Runtime Minutes': 5.926780290416658},
 {'Feature Set': 'Original + Graph Features',
  'Model': 'GraphSAGE',
  'Num Features': 169,
  'Accuracy': 0.9702880058330295,
  'Illicit Precision': 0.8717277486910995,
  'Illicit Recall': 0.7449664429530202,
  'Illicit F1': 0.8033775633293124,
  'Runtime Seconds': 535.5016962000009,
  'Runtime Minutes': 8.925028270000015},
 {'Feature Set': 'Original Featur

In [89]:
# combine everything
comparison_df = pd.concat(
    [
        comparison_03,
        comparison_gnn_results
    ],
    ignore_index=True
)

In [90]:
# same display logic
comparison_display = comparison_df[
    [
        'Feature Set',
        'Model',
        'Num Features',
        'Accuracy',
        'Illicit Precision',
        'Illicit Recall',
        'Illicit F1',
        'Runtime Minutes'
    ]
].copy()

comparison_display = comparison_display.round({
    "Accuracy": 6,
    'Illicit Precision': 6,
    "Illicit Recall": 6,
    "Illicit F1": 6,
    "Runtime Minutes": 3
})


In [94]:
comparison_display = comparison_display.sort_values(by='Illicit F1', ascending=False).reset_index(drop=True)
display(comparison_display)

,Feature Set,Model,Num Features,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Runtime Minutes
0,Original Features,Random Forest,169,0.990339,0.994975,0.885906,0.937278,0.694
1,Node2Vec Embeddings,Random Forest,201,0.989245,0.989899,0.876957,0.930012,0.933
2,DeepWalk Embeddings,Random Forest,201,0.989245,0.992386,0.874720,0.929845,0.927
3,Original + Graph Features,GraphSAGE,169,0.971382,0.909605,0.720358,0.803995,8.638
4,Original Features Only,GraphSAGE,165,0.968647,0.874659,0.718121,0.788698,8.635
5,Original + Graph Features,GCN,169,0.952971,0.853933,0.510067,0.638655,6.133
6,Original Features Only,GCN,165,0.948232,0.802974,0.483221,0.603352,5.820
7,Node2Vec Embeddings,Logistic Regression,201,0.784360,0.269424,0.961969,0.420950,0.094
8,Original Features,Logistic Regression,169,0.784360,0.268553,0.955257,0.419244,0.176
9,DeepWalk Embeddings,Logistic Regression,201,0.770142,0.255995,0.955257,0.403783,0.102


The updated comparison table now includes the performance of GCN and GraphSAGE with both feature sets (original + graph features, and original features only).

Key Observations from the full comparison table:

-   **Impact of Graph Features:** For GCN, including hand-crafted graph features ('Original + Graph Features') led to a slight improvement in illicit F1 score compared to using 'Original Features Only'. For GraphSAGE, the performance between the two feature sets is very similar, suggesting that GraphSAGE's aggregation mechanism might be effectively learning similar structural information, or that the hand-crafted features offer less distinct value for this model.
-   **GNN vs. Baselines:** While GraphSAGE outperforms GCN, none of the static GNNs (GCN or GraphSAGE) with either feature set are able to surpass the Random Forest model using 'Original Features' in terms of illicit F1 score or overall accuracy on the validation data. This reinforces the earlier finding that engineered graph features combined with classical ML remain highly competitive for this dataset.
-   **Runtime:** GNNs have significantly longer runtimes compared to Random Forest and Logistic Regression models, especially when considering the need for GPU acceleration for larger graphs.

This comprehensive comparison further highlights the challenge of outperforming strong baseline models that leverage well-engineered features, even with advanced GNN architectures. It also underscores the potential need for temporal GNNs to truly capture the dynamic nature of the transaction network.

In [92]:
# save it
comparison_display.to_csv("/content/drive/MyDrive/Elliptic/results/04_static_gnn_comparison.csv",
    index=False
)

### Final Test Set Evaluation

Now that the GNN models (GCN and GraphSAGE) have been developed and compared using the validation set, we will perform a final, unbiased evaluation on the **test set**. This will provide the definitive performance metrics for how well these models generalize to unseen future transactions.

In [93]:
epochs = 200

print('--- GCN Final Test Set Evaluation (Original + Graph Features) ---')
gcn_model_final_full = GCN(in_channels=data_full.x.shape[1])
optimizer_gcn_full = torch.optim.Adam(gcn_model_final_full.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(1, epochs + 1):
    gcn_model_final_full.train()
    optimizer_gcn_full.zero_grad()
    out_gcn = gcn_model_final_full(data_full)
    loss_gcn = F.cross_entropy(out_gcn[train_mask_full], data_full.y[train_mask_full])
    loss_gcn.backward()
    optimizer_gcn_full.step()

gcn_model_final_full.eval()
with torch.no_grad():
    out_gcn_test = gcn_model_final_full(data_full)
    test_preds_gcn_full = out_gcn_test[test_mask_full].argmax(dim=1).cpu().numpy()
    y_test_gcn_full = data_full.y[test_mask_full].cpu().numpy()

print(classification_report(
    y_test_gcn_full,
    test_preds_gcn_full,
    target_names=['licit', 'illicit']
))
print(confusion_matrix(
    y_test_gcn_full,
    test_preds_gcn_full
))

print('\n--- GCN Final Test Set Evaluation (Original Features Only) ---')
gcn_model_final_no_graph = GCN(in_channels=data_no_graph.x.shape[1])
optimizer_gcn_no_graph = torch.optim.Adam(gcn_model_final_no_graph.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(1, epochs + 1):
    gcn_model_final_no_graph.train()
    optimizer_gcn_no_graph.zero_grad()
    out_gcn = gcn_model_final_no_graph(data_no_graph)
    loss_gcn = F.cross_entropy(out_gcn[train_mask_no_graph], data_no_graph.y[train_mask_no_graph])
    loss_gcn.backward()
    optimizer_gcn_no_graph.step()

gcn_model_final_no_graph.eval()
with torch.no_grad():
    out_gcn_test = gcn_model_final_no_graph(data_no_graph)
    test_preds_gcn_no_graph = out_gcn_test[test_mask_no_graph].argmax(dim=1).cpu().numpy()
    y_test_gcn_no_graph = data_no_graph.y[test_mask_no_graph].cpu().numpy()

print(classification_report(
    y_test_gcn_no_graph,
    test_preds_gcn_no_graph,
    target_names=['licit', 'illicit']
))
print(confusion_matrix(
    y_test_gcn_no_graph,
    test_preds_gcn_no_graph
))

print('\n--- GraphSAGE Final Test Set Evaluation (Original + Graph Features) ---')
sage_model_final_full = GraphSAGE(in_channels=data_full.x.shape[1])
optimizer_sage_full = torch.optim.Adam(sage_model_final_full.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(1, epochs + 1):
    sage_model_final_full.train()
    optimizer_sage_full.zero_grad()
    out_sage = sage_model_final_full(data_full)
    loss_sage = F.cross_entropy(out_sage[train_mask_full], data_full.y[train_mask_full])
    loss_sage.backward()
    optimizer_sage_full.step()

sage_model_final_full.eval()
with torch.no_grad():
    out_sage_test = sage_model_final_full(data_full)
    test_preds_sage_full = out_sage_test[test_mask_full].argmax(dim=1).cpu().numpy()
    y_test_sage_full = data_full.y[test_mask_full].cpu().numpy()

print(classification_report(
    y_test_sage_full,
    test_preds_sage_full,
    target_names=['licit', 'illicit']
))
print(confusion_matrix(
    y_test_sage_full,
    test_preds_sage_full
))

print('\n--- GraphSAGE Final Test Set Evaluation (Original Features Only) ---')
sage_model_final_no_graph = GraphSAGE(in_channels=data_no_graph.x.shape[1])
optimizer_sage_no_graph = torch.optim.Adam(sage_model_final_no_graph.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(1, epochs + 1):
    sage_model_final_no_graph.train()
    optimizer_sage_no_graph.zero_grad()
    out_sage = sage_model_final_no_graph(data_no_graph)
    loss_sage = F.cross_entropy(out_sage[train_mask_no_graph], data_no_graph.y[train_mask_no_graph])
    loss_sage.backward()
    optimizer_sage_no_graph.step()

sage_model_final_no_graph.eval()
with torch.no_grad():
    out_sage_test = sage_model_final_no_graph(data_no_graph)
    test_preds_sage_no_graph = out_sage_test[test_mask_no_graph].argmax(dim=1).cpu().numpy()
    y_test_sage_no_graph = data_no_graph.y[test_mask_no_graph].cpu().numpy()

print(classification_report(
    y_test_sage_no_graph,
    test_preds_sage_no_graph,
    target_names=['licit', 'illicit']
))
print(confusion_matrix(
    y_test_sage_no_graph,
    test_preds_sage_no_graph
))


--- GCN Final Test Set Evaluation (Original + Graph Features) ---
              precision    recall  f1-score   support

       licit       0.96      0.99      0.98     10548
     illicit       0.76      0.29      0.42       636

    accuracy                           0.95     11184
   macro avg       0.86      0.64      0.70     11184
weighted avg       0.95      0.95      0.94     11184

[[10488    60]
 [  450   186]]

--- GCN Final Test Set Evaluation (Original Features Only) ---
              precision    recall  f1-score   support

       licit       0.96      1.00      0.98     10548
     illicit       0.88      0.26      0.40       636

    accuracy                           0.96     11184
   macro avg       0.92      0.63      0.69     11184
weighted avg       0.95      0.96      0.94     11184

[[10525    23]
 [  473   163]]

--- GraphSAGE Final Test Set Evaluation (Original + Graph Features) ---
              precision    recall  f1-score   support

       licit       0.97   

The final test set produced lower performance than the validation set for both static GNN architectures, indicating that generalizing to later time steps remains challenging.

GCN achieved an illicit F1 score of approximately 0.42 when using the engineered graph features and 0.40 when using only the original transaction features. This suggests that the engineered graph features provided a small performance benefit for the GCN model.

GraphSAGE achieved an illicit F1 score of approximately 0.58 with the engineered graph features and 0.59 using only the original transaction features. The difference is small, but it suggests that GraphSAGE is capable of learning much of the structural information directly through neighborhood aggregation, reducing its dependence on hand-engineered graph features.

Although GraphSAGE consistently outperformed GCN, neither static GNN surpassed the Random Forest baseline from the previous notebook. These results suggest that basic static message-passing architectures alone are insufficient to outperform the strongest classical machine learning baseline on this dataset.

## Summary

This notebook evaluated two static Graph Neural Network architectures, GCN and GraphSAGE, on the Elliptic Bitcoin transaction network. Although the graph was treated as a single static network, temporal train, validation, and test masks were used throughout the evaluation to preserve chronological ordering.

GraphSAGE consistently outperformed GCN on both the validation and final test sets. On the final test set, GCN achieved an illicit F1 score of approximately 0.42 with the engineered graph features, while GraphSAGE achieved approximately 0.58. An ablation study also showed that removing the engineered graph features had little effect on GraphSAGE performance, suggesting that the model can learn much of the relevant structural information through message passing.

Despite these improvements, neither static GNN architecture outperformed the Random Forest baseline from the previous notebook. This demonstrates that increased model complexity does not necessarily lead to better fraud detection performance and reinforces the importance of comparing advanced graph models against strong classical baselines.

These findings motivate the next stage of the project: investigating temporal graph neural networks, which may better capture the evolving transaction patterns present in the Elliptic Bitcoin dataset.

## Next step

GCN and GraphSAGE both learn node representations from graph structure and node features, but the models in this notebook treat the Elliptic transaction network as a static graph. Since the dataset is organized across 49 time steps, a natural next step is to investigate temporal graph methods.

The next notebook will explore whether temporal GNN approaches can better capture how transaction behavior changes over time and whether this improves illicit transaction detection compared with static GNNs and classical machine learning baselines.